In [0]:
# =============================================================================
# CATÁLOGO DE DADOS - Documentação no Unity Catalog
# =============================================================================
# Objetivo: Aplicar descrições (COMMENT) nas tabelas e colunas do pipeline,
# tornando o Unity Catalog a fonte oficial de documentação dos dados.
# Os comandos são executados individualmente via spark.sql() para evitar
# limitações de parsing de múltiplos statements em uma única célula SQL.
# =============================================================================

# -----------------------------------------------------------------------------
# Descrições em nível de TABELA
# -----------------------------------------------------------------------------
tabelas = {
    # --- Camada Bronze ---
    "bronze_orders": "Camada Bronze. Pedidos brutos do dataset Olist, ingeridos sem transformação a partir de olist_orders_dataset.csv. Contém status e marcos temporais de cada pedido.",
    "bronze_order_items": "Camada Bronze. Itens de pedido brutos, ingeridos de olist_order_items_dataset.csv. Granularidade: um registro por item dentro de um pedido.",
    "bronze_order_payments": "Camada Bronze. Pagamentos brutos, ingeridos de olist_order_payments_dataset.csv. Um pedido pode ter múltiplos registros de pagamento.",
    "bronze_order_reviews": "Camada Bronze. Avaliações brutas, ingeridas de olist_order_reviews_dataset.csv com opção multiLine habilitada para preservar comentários com quebras de linha.",
    "bronze_products": "Camada Bronze. Produtos brutos, ingeridos de olist_products_dataset.csv. Inclui categoria e atributos físicos.",
    "bronze_customers": "Camada Bronze. Clientes brutos, ingeridos de olist_customers_dataset.csv. Cada customer_id representa um cliente em um pedido específico.",
    "bronze_sellers": "Camada Bronze. Vendedores brutos, ingeridos de olist_sellers_dataset.csv.",
    "bronze_category_translation": "Camada Bronze. Tabela de apoio com tradução das categorias de produto de português para inglês.",
    "bronze_geolocation": "Camada Bronze. Coordenadas geográficas por CEP. Ingerida para completude do dataset, porém não utilizada nas camadas Silver e Gold por estar fora do escopo das perguntas de negócio deste MVP.",

    # --- Camada Silver ---
    "silver_orders": "Camada Silver. Pedidos limpos: datas convertidas de string para timestamp, registros sem status removidos e duplicatas eliminadas por order_id.",
    "silver_order_items": "Camada Silver. Itens limpos: preço e frete tipados como double, registros com preço nulo ou negativo removidos, duplicatas eliminadas pela chave composta order_id mais order_item_id.",
    "silver_order_payments": "Camada Silver. Pagamentos limpos: valores tipados, registros negativos removidos, duplicatas eliminadas por order_id mais payment_sequential.",
    "silver_order_reviews": "Camada Silver. Avaliações limpas: review_score convertido com try_cast para tolerar valores malformados na origem, restrito ao domínio válido de 1 a 5.",
    "silver_products": "Camada Silver. Produtos limpos: registros sem categoria definida removidos por não contribuírem às análises por categoria.",
    "silver_customers": "Camada Silver. Clientes limpos: siglas de estado padronizadas para maiúsculo, evitando inconsistências de escrita.",
    "silver_sellers": "Camada Silver. Vendedores limpos: siglas de estado padronizadas para maiúsculo.",
    "silver_category_translation": "Camada Silver. Tradução de categorias com registros sem tradução disponível removidos.",

    # --- Camada Gold ---
    "gold_fato_pedidos": "Camada Gold. Tabela fato central do MVP. Granularidade: um registro por item de pedido. Construída pelo join entre silver_order_items, silver_orders, silver_customers, silver_products, silver_category_translation, silver_sellers e a agregação de silver_order_payments por pedido. Suporta todas as análises de padrão de compra do objetivo.",
}

for tabela, descricao in tabelas.items():
    spark.sql(f"COMMENT ON TABLE workspace.default.{tabela} IS '{descricao}'")
    print(f"✓ Descrição aplicada: {tabela}")

# -----------------------------------------------------------------------------
# Descrições em nível de COLUNA na tabela Gold
# -----------------------------------------------------------------------------
colunas_gold = {
    "order_id": "Identificador único do pedido. Origem: silver_orders.order_id. Hash alfanumérico de 32 caracteres.",
    "order_item_id": "Sequência do item dentro do pedido. Origem: silver_order_items.order_item_id. Domínio: inteiro de 1 a N.",
    "customer_id": "Identificador do cliente no pedido. Origem: silver_orders.customer_id. Hash alfanumérico.",
    "product_id": "Identificador único do produto. Origem: silver_order_items.product_id. Hash alfanumérico.",
    "seller_id": "Identificador único do vendedor. Origem: silver_order_items.seller_id. Hash alfanumérico.",
    "order_purchase_timestamp": "Data e hora em que a compra foi realizada. Origem: silver_orders, convertido de string para timestamp na camada Silver. Domínio: 2016-09-04 a 2018-09-03.",
    "ano_compra": "Ano da compra. Derivado: extraído de order_purchase_timestamp via função year. Domínio: 2016, 2017, 2018.",
    "mes_compra": "Mês da compra. Derivado: extraído de order_purchase_timestamp via função month. Domínio: 1 a 12.",
    "order_status": "Situação do pedido. Origem: silver_orders.order_status. Domínio: delivered, shipped, canceled, unavailable, invoiced, processing, created, approved.",
    "categoria_pt": "Categoria do produto em português. Origem: silver_products.product_category_name, obtido por join com product_id. Domínio: 73 categorias distintas.",
    "categoria_en": "Categoria do produto em inglês. Origem: silver_category_translation, obtido por join com categoria_pt. Domínio: 71 categorias distintas.",
    "preco_item": "Preço unitário do item, sem frete. Origem: silver_order_items.price. Domínio: R$ 0,85 a R$ 6.735,00.",
    "frete": "Valor do frete atribuído ao item. Origem: silver_order_items.freight_value. Domínio: R$ 0,00 a R$ 409,68. Valor zero é válido e representa frete grátis ou promoção.",
    "valor_total_item": "Valor total do item. Derivado: soma de preco_item e frete, arredondado a duas casas decimais.",
    "forma_pagamento": "Forma de pagamento principal do pedido. Origem: silver_order_payments, agregado por order_id. Domínio: credit_card, boleto, voucher, debit_card, not_defined.",
    "parcelas": "Número médio de parcelas do pedido. Derivado: média de payment_installments agregada por order_id e arredondada para inteiro. Domínio: 1 a 24.",
    "valor_total_pedido": "Valor total efetivamente pago no pedido. Derivado: soma de payment_value de todos os pagamentos do pedido em silver_order_payments.",
    "estado_cliente": "Unidade federativa do cliente comprador. Origem: silver_customers.customer_state, padronizado para maiúsculo. Domínio: 27 siglas de estados brasileiros.",
    "cidade_cliente": "Cidade do cliente comprador. Origem: silver_customers.customer_city. Texto livre em minúsculo conforme origem.",
    "estado_vendedor": "Unidade federativa do vendedor. Origem: silver_sellers.seller_state, padronizado para maiúsculo. Domínio: 27 siglas de estados brasileiros.",
}

print("\n--- Aplicando descrições nas colunas da gold_fato_pedidos ---")
for coluna, descricao in colunas_gold.items():
    spark.sql(
        f"ALTER TABLE workspace.default.gold_fato_pedidos "
        f"ALTER COLUMN {coluna} COMMENT '{descricao}'"
    )
    print(f"✓ {coluna}")

print("\nCatálogo de dados documentado com sucesso no Unity Catalog.")